In [1]:
import json
import pickle
import faiss
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer


c:\Users\USER\Desktop\MediRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMBEDDING_MODEL_NAME = "pritamdeka/S-BioBert-snli-multinli-stsb"
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

In [15]:
DATA_PATH = "../data division logic/miriad_neurology.jsonl"

records = [] 
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# with open(DATA_PATH, "r", encoding="utf-8") as f:
#     for i, line in enumerate(f):
#         if i >= 5000:
#             break
#         records.append(json.loads(line))

print(f"Loaded {len(records)} records")

Loaded 205154 records


In [16]:
texts_to_embed = [r["answer"] for r in records]

In [17]:
embeddings = embedder.encode(
    texts_to_embed,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings).astype("float32")

Batches: 100%|██████████| 12823/12823 [8:05:37<00:00,  2.27s/it]     


In [18]:
dimension = embeddings.shape[1]
 
index = faiss.IndexFlatIP(dimension)  # Inner Product = cosine (after normalization)
index.add(embeddings)

print(f"FAISS index contains {index.ntotal} vectors")

FAISS index contains 205154 vectors


In [19]:
metadata = []

for r in records:
    metadata.append({
        "qa_id": r["qa_id"],
        "answer": r["answer"],
        "passage_text": r["passage_text"],
        "paper_id": r.get("paper_id")
    })

In [20]:
faiss.write_index(index, "embeddings/faiss.index")

with open("embeddings/metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Vector DB successfully built and saved.")

Vector DB successfully built and saved.


In [3]:
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from nltk.tokenize import word_tokenize
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
FAISS_INDEX_PATH = "embeddings/faiss.index"
METADATA_PATH = "embeddings/metadata.pkl"

index = faiss.read_index(FAISS_INDEX_PATH)

with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

assert index.ntotal == len(metadata)
print(f"Loaded FAISS index with {index.ntotal} vectors")


Loaded FAISS index with 205154 vectors


In [5]:
import torch

MODEL_DIR = "../chunk-rag pipeline/flan_t5_neuro_rewriter"  # <-- your local directory

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Query rewriter loaded")

Query rewriter loaded


In [6]:
def rewrite_query_for_bm25(user_query: str) -> str:
    """
    Rewrite a user query for BM25 retrieval using a local T5 model.
    """
    import torch
    prompt = "rewrite query: " + user_query
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    # Move tensors to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=4,
            early_stopping=True
        )
    rewritten = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return rewritten


In [7]:
import os
from rank_bm25 import BM25Okapi

bm25_path = "embeddings/bm25.pkl"

if os.path.exists(bm25_path):
    with open(bm25_path, "rb") as f:
        bm25 = pickle.load(f)
    print("BM25 index loaded")
else:
    corpus = [word_tokenize(item["answer"].lower()) for item in metadata]
    bm25 = BM25Okapi(corpus)
    with open(bm25_path, "wb") as f:
        pickle.dump(bm25, f)
    print("BM25 index built and saved")


BM25 index loaded


In [8]:
def bm25_retrieve(query: str, top_k: int = 50):
    tokenized_query = word_tokenize(query.lower())
    scores = bm25.get_scores(tokenized_query)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []
    for rank, idx in enumerate(ranked_indices, start=1):
        record = metadata[idx].copy()
        record["bm25_rank"] = rank
        record["bm25_score"] = float(scores[idx])
        results.append(record)

    return results

In [9]:
def faiss_retrieve(query: str, top_k: int = 20):
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        record = metadata[idx].copy()
        record["faiss_rank"] = rank
        record["faiss_score"] = float(score)
        results.append(record)

    return results

In [10]:
def reciprocal_rank_fusion(faiss_results, bm25_results, k: int = 60):
    scores = defaultdict(float)
    records = {}

    for r in faiss_results:
        doc_id = r["qa_id"]
        scores[doc_id] += 1.0 / (k + r["faiss_rank"])
        records[doc_id] = r

    for r in bm25_results:
        doc_id = r["qa_id"]
        scores[doc_id] += 1.0 / (k + r["bm25_rank"])
        records[doc_id] = r

    fused = []
    for doc_id, score in scores.items():
        record = records[doc_id].copy()
        record["rrf_score"] = score
        fused.append(record)

    fused.sort(key=lambda x: x["rrf_score"], reverse=True)
    return fused

In [11]:
def hybrid_retrieve(
    user_query: str,
    top_k: int = 5
):
    # Semantic retrieval uses ORIGINAL query
    faiss_results = faiss_retrieve(user_query, top_k=40)

    # Lexical retrieval uses REWRITTEN query
    rewritten_query = rewrite_query_for_bm25(user_query)
    bm25_results = bm25_retrieve(rewritten_query, top_k=40)

    fused_results = reciprocal_rank_fusion(
        faiss_results,
        bm25_results,
        k=60
    )

    return fused_results[:top_k], rewritten_query


In [12]:
user_query = (
    "What is cortical superficial siderosis (cSS) and how does it relate to cerebral amyloid angiopathy (CAA)?"
)

results, rewritten_query = hybrid_retrieve(user_query, top_k=5)

print("Rewritten (BM25) query:", rewritten_query)

for i, r in enumerate(results, 1):
    print(f"\nResult {i}")
    print("RRF score:", r["rrf_score"])
    print("Answer:", r["answer"])

Rewritten (BM25) query: cortical superficial siderosis cerebral amyloid angiopathy

Result 1
RRF score: 0.032018442622950824
Answer: Cortical superficial siderosis (cSS) is a hemorrhagic signature of cerebral amyloid angiopathy (CAA), which is a common small vessel disease characterized by amyloid deposition in the brain's blood vessels. cSS occurs when there are bleeding episodes within or near the cortical sulci, likely originating from amyloid-laden arterioles in the superficial cortex and leptomeninges. cSS is a common manifestation of CAA, found in 40%-60% of patients, and has distinct clinical and prognostic implications in this context.

Result 2
RRF score: 0.030834914611005692
Answer: cSS stands for cortical superficial siderosis, which is hemosiderin deposition in the subpial superficial cortical layers. In patients with CAA (cerebral amyloid angiopathy), the presence of cSS on T2*-GRE MRI significantly increases the risk of future symptomatic lobar ICH (intracerebral hemorrha

In [ ]:
# import json

# # Load the 50-row dataset
# with open("../data division logic/miriad_neurology_first50.json", "r", encoding="utf-8") as f:
#     eval_records = json.load(f)

# top_k = 10
# first_is_answer = 0
# answer_in_topk = 0

# for row in eval_records:
#     user_query = row["question"]
#     expected_answer = row["answer"]
#     results, _ = hybrid_retrieve(user_query, top_k=top_k)
#     answers = [r["answer"] for r in results]
#     if answers and answers[0] == expected_answer:
#         first_is_answer += 1
#     if expected_answer in answers:
#         answer_in_topk += 1

# print(f"Number of times the first result is the expected answer: {first_is_answer} out of {len(eval_records)}")
# print(f"Number of times the expected answer appears in top {top_k}: {answer_in_topk} out of {len(eval_records)}")

Number of times the first result is the expected answer: 25 out of 50
Number of times the expected answer appears in top 10: 43 out of 50


In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv(r"C:\Users\USER\Desktop\MediRAG\fine tuning re-writer\.env")
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

print("OpenAI client initialized")


OpenAI client initialized


In [15]:
SYSTEM_PROMPT = """
You are a medical question answering assistant.

You are given:
1) A user question
2) A small set of retrieved answers from medical literature

IMPORTANT:
- Some retrieved answers may be more directly relevant to the question than others.

STRICT RULES:
- You MUST ONLY use information present in the retrieved answers.
- You MUST NOT add new facts, interpretations, or explanations.
- You MUST NOT speculate or generalize.
- You MUST NOT combine information that is not explicitly supported by the retrieved answers.
- If the retrieved answers do not contain sufficient information to answer the question,
  respond with exactly:
  "The retrieved sources do not contain sufficient information to answer this question."

Your task is ONLY to:
- Identify which retrieved answer(s) are most relevant to the user question
- Rephrase and, if appropriate, combine ONLY the relevant answer(s)
- Match the wording and intent of the user's question
- Produce a clear, concise final answer
"""


In [16]:
def format_retrieved_answers(results):
    formatted = []
    for i, r in enumerate(results, 1):
        formatted.append(f"[Source {i}] {r['answer']}")
    return "\n\n".join(formatted)


In [17]:
def llm_rephrase_answer(user_query, retrieved_results):
    """
    Uses OpenAI LLM as a STRICT rephraser.
    """

    evidence = format_retrieved_answers(retrieved_results)

    user_prompt = f"""
User Question:
{user_query}

Retrieved Answers:
{evidence}

Final Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",   # fast + reliable for rephrasing
        temperature=0.0,        # CRITICAL: no creativity
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content.strip()


In [18]:
def rag_answer(user_query, top_k=5):
    # Step 1: Hybrid retrieval (already implemented)
    retrieved_results, rewritten_query = hybrid_retrieve(
        user_query,
        top_k=top_k
    )

    # Step 2: LLM rephrasing
    final_answer = llm_rephrase_answer(
        user_query,
        retrieved_results
    )

    return {
        "user_query": user_query,
        "retrieved_answers": retrieved_results,
        "final_answer": final_answer
    }

In [22]:
user_query = (
    "How is the Glasgow Coma Scale score determined in patients with intracranial hemorrhage?"
)

output = rag_answer(user_query, top_k=10)

print("USER QUERY:")
print(output["user_query"])

print("\nFINAL ANSWER:")
print(output["final_answer"])

USER QUERY:
How is the Glasgow Coma Scale score determined in patients with intracranial hemorrhage?

FINAL ANSWER:
The Glasgow Coma Scale (GCS) score in patients with intracranial hemorrhage is determined at the initial evaluation using all available clinical information. If patients are described as being "in coma" or "unresponsive" without additional discriminating information, they are assigned a score of 7. When seizures are associated with the clinical presentation, the best estimated GCS score before the seizure or after complete recovery from the postictal state is used.
